# Onamiz Cry Classifier — EDA

Donate-a-Cry corpus tahlili: audio sifati, klass balansi, spektrogramlar.

**Maqsad:** Dataset xususiyatlarini tushunish va modeling strategiyasini aniqlash.

In [ ]:
import os
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

DATA_DIR = Path('../data/donateacry-corpus/donateacry_corpus_cleaned_and_updated_data')
CLASSES = ['hungry', 'tired', 'discomfort', 'burping', 'belly_pain']
print('Dataset:', DATA_DIR.resolve())

## 1. Klass balansi

In [ ]:
rows = []
for cls in CLASSES:
    files = list((DATA_DIR / cls).glob('*.wav'))
    for f in files:
        rows.append({'class': cls, 'path': str(f), 'size_kb': f.stat().st_size / 1024})

df = pd.DataFrame(rows)
print(f'Jami: {len(df)} ta audio fayl\n')

counts = df['class'].value_counts().reindex(CLASSES)
pct = (counts / counts.sum() * 100).round(1)
summary = pd.DataFrame({'count': counts, 'pct': pct})
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#D86080', '#7B3FB0', '#FF9F40', '#4CAF50', '#E91E63']
bars = ax.bar(counts.index, counts.values, color=colors)
for bar, count in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{count}', ha='center', fontweight='bold')
ax.set_title('Klass balansi — Donate-a-Cry', fontsize=14, fontweight='bold')
ax.set_ylabel('Sample soni')
plt.tight_layout()
plt.show()

print('\n⚠️  Imbalanced — hungry dominates')
print(f'    Ratio: hungry / burping = {counts["hungry"] / counts["burping"]:.1f}x')

## 2. Audio xususiyatlari

Har bir audio uchun: davomiyligi, sample rate, RMS energiya.

In [ ]:
def audio_stats(path):
    y, sr = librosa.load(path, sr=None)
    return {
        'duration': len(y) / sr,
        'sample_rate': sr,
        'rms': float(np.sqrt(np.mean(y**2))),
        'max_amp': float(np.max(np.abs(y))),
    }

stats = df['path'].apply(audio_stats).apply(pd.Series)
df_stats = pd.concat([df, stats], axis=1)
df_stats.head()

In [ ]:
print('Sample rate distribution:')
print(df_stats['sample_rate'].value_counts())
print('\nDuration statistika (sekund):')
print(df_stats.groupby('class')['duration'].describe()[['mean', 'min', 'max', 'std']].round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for cls in CLASSES:
    sub = df_stats[df_stats['class'] == cls]
    axes[0].hist(sub['duration'], bins=15, alpha=0.6, label=cls)
axes[0].set_xlabel('Davomiylik (sekund)')
axes[0].set_ylabel('Sample soni')
axes[0].set_title('Audio davomiyligi taqsimoti')
axes[0].legend()

sns.boxplot(data=df_stats, x='class', y='rms', ax=axes[1], order=CLASSES)
axes[1].set_title('RMS energiya — klass bo\'yicha')
axes[1].set_ylabel('RMS')

plt.tight_layout()
plt.show()

## 3. Spektrogramlar — har klassdan bittadan

In [ ]:
def plot_mel_spectrogram(path, title, ax):
    y, sr = librosa.load(path, sr=16000)  # YAMNet'ga moslab 16kHz
    mel = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel',
                                    ax=ax, fmax=8000)
    ax.set_title(title, fontweight='bold')
    return img

fig, axes = plt.subplots(5, 1, figsize=(12, 14))
for ax, cls in zip(axes, CLASSES):
    sample = df_stats[df_stats['class'] == cls].iloc[0]
    plot_mel_spectrogram(sample['path'], f'{cls} — {sample["duration"]:.1f}s', ax)

plt.tight_layout()
plt.show()

## 4. Class weights va augmentation rejasi

Imbalanced datasetni qanday tuzatamiz:

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

y_labels = df['class'].values
weights = compute_class_weight('balanced', classes=np.array(CLASSES), y=y_labels)
class_weights = dict(zip(CLASSES, weights.round(2)))

print('Class weights (balanced):')
for cls, w in class_weights.items():
    print(f'  {cls:12s} → weight = {w}')

print('\n📌 Augmentation target (kichik klasslar):')
target_per_class = 100
for cls in CLASSES:
    cur = counts[cls]
    needed = max(0, target_per_class - cur)
    mult = round(target_per_class / cur, 1) if cur > 0 else 0
    print(f'  {cls:12s} → +{needed} ta kerak ({mult}x augmentation)')

## 5. Xulosalar va keyingi qadamlar

**Aniqlangan muammolar:**
1. ⚠️ Imbalanced — `hungry` 83.6%, `burping` 1.7%
2. 🎵 Sample rate 8000 Hz — YAMNet 16000 Hz uchun resample kerak
3. 📏 Davomiyligi har xil — 3-7 sekund oralig'ida (trim/pad 3s ga)

**Strategiya:**
- ✅ Stratified train/val/test split (70/15/15)
- ✅ Class weights balanced
- ✅ Audio augmentation (pitch, time, noise) — kam klasslarni 5-10x kengaytirish
- ✅ YAMNet feature extractor (frozen) + kichik classifier head
- ✅ Focal loss (qiyin sample'larga e'tibor)

**Keyingi notebook:** `02_train.ipynb`